# SAE-анализ для детекции галлюцинаций
## Hiromi — часть 3: Sparse Autoencoder + Mechanistic Interpretability

В этом notebook мы реализуем **SAE-анализ галлюцинаций** поверх результатов probing-классификатора (часть 2).

**Идея**: разложить активации лучшего слоя Gemma 2 2B через предобученный SAE (Gemma Scope) на ~16k интерпретируемых фичей и найти те, которые специфически активируются на галлюцинациях.

**Результаты предыдущих частей**:
- Часть 1 (LLM-as-a-Judge): Self-Consistency EN = **75.77%** ← baseline
- Часть 2 (Probing): Linear probe на hidden states Gemma 2 2B
- Часть 3 (текущая): SAE-фичи → интерпретируемые «галлюцинационные» признаки

## 1. Установка и импорты

In [ ]:
# Устанавливаем зависимости
!pip install -q transformers datasets accelerate sentencepiece protobuf
!pip install -q scikit-learn matplotlib seaborn tqdm scipy
!pip install -q sae-lens huggingface_hub safetensors

In [ ]:
import os
import gc
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from scipy import stats

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Фиксируем seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Единый стиль графиков
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
})
sns.set_theme(style='whitegrid', palette='muted')

## 2. Конфигурация

In [ ]:
# ============================================================
# КОНФИГУРАЦИЯ — меняй эти параметры при необходимости
# ============================================================

# Лучший слой из probing-части (часть 2)
# Если у тебя есть результаты probing — подставь свой слой
BEST_LAYER = 13                # слой модели (0-индексация, 0 = embedding)

# Стратегия агрегации токенов — та, что лучше всего сработала в probing
# Варианты: 'last_token', 'mean_pooling', 'answer_mean'
AGG_STRATEGY = 'last_token'

# Параметры модели и данных
MODEL_NAME = 'google/gemma-2-2b'
N_SAMPLES_PER_CLASS = 500      # 500 correct + 500 hallucination
BATCH_SIZE = 8
MAX_LENGTH = 512

# SAE конфигурация (Gemma Scope)
SAE_RELEASE = 'gemma-scope-2b-pt-res'
SAE_WIDTH = '16k'              # ширина SAE: 16k фичей
SAE_L0 = 71                    # средний L0 (разреженность)

# Пути к файлам
HIDDEN_STATES_PATH = 'hidden_states.npz'   # из probing-части
SAE_ACTIVATIONS_PATH = 'sae_activations.npz'

# Результаты предыдущих частей для итогового сравнения
PART1_RESULTS = {
    'Zero-shot RU':         0.7492,
    'Zero-shot EN':         0.7389,
    'Few-shot RU':          0.7467,
    'Few-shot EN':          0.7512,
    'CoT RU':               0.7423,
    'CoT EN':               0.7534,
    'Self-Cons. RU':        0.7467,
    'Self-Cons. EN':        0.7577,
    'Reference-based':      0.8932,
}
PART2_BEST_LINEAR = 0.80       # замени на реальный результат из probing

BASELINE_ACC = 0.7577          # Self-Consistency EN

print('Конфигурация:')
print(f'  BEST_LAYER:   {BEST_LAYER}')
print(f'  AGG_STRATEGY: {AGG_STRATEGY}')
print(f'  SAE ширина:   {SAE_WIDTH} фичей')
print(f'  Базовый датасет: {N_SAMPLES_PER_CLASS * 2} примеров')

## 3. Загрузка данных TruthfulQA

Используем **те же 1000 примеров** (500 correct + 500 hallucination, random_state=42), что и в probing-части.

In [ ]:
print('Загружаем TruthfulQA...')
dataset = load_dataset('truthfulqa/truthful_qa', 'generation', split='validation')
print(f'Всего вопросов: {len(dataset)}')

def build_pairs(dataset):
    """Разворачивает вопросы в пары (вопрос, ответ, метка, категория)."""
    correct_pairs = []
    hallucination_pairs = []

    for item in dataset:
        question = item['question']
        category = item.get('category', 'Unknown')

        for ans in item['correct_answers']:
            ans = ans.strip()
            if ans:
                correct_pairs.append({
                    'text': f"Question: {question}\nAnswer: {ans}",
                    'question': question,
                    'answer': ans,
                    'label': 1,
                    'label_name': 'correct',
                    'category': category,
                })

        for ans in item['incorrect_answers']:
            ans = ans.strip()
            if ans:
                hallucination_pairs.append({
                    'text': f"Question: {question}\nAnswer: {ans}",
                    'question': question,
                    'answer': ans,
                    'label': 0,
                    'label_name': 'hallucination',
                    'category': category,
                })

    return correct_pairs, hallucination_pairs


correct_pairs, hallucination_pairs = build_pairs(dataset)
print(f'Correct пар:       {len(correct_pairs)}')
print(f'Hallucination пар: {len(hallucination_pairs)}')

In [ ]:
# Стратифицированный сэмпл: 500 correct + 500 hallucination
rng = random.Random(SEED)

sampled_correct       = rng.sample(correct_pairs, N_SAMPLES_PER_CLASS)
sampled_hallucination = rng.sample(hallucination_pairs, N_SAMPLES_PER_CLASS)

all_samples = sampled_correct + sampled_hallucination
rng.shuffle(all_samples)

texts      = [s['text']      for s in all_samples]
labels     = np.array([s['label']    for s in all_samples])
questions  = [s['question']  for s in all_samples]
categories = [s['category']  for s in all_samples]

print(f'Итого: {len(texts)} примеров')
print(f'  correct:       {labels.sum()}')
print(f'  hallucination: {(labels == 0).sum()}')
print(f'\nКатегорий: {len(set(categories))}')
print('Топ-5 по количеству:')
from collections import Counter
for cat, cnt in Counter(categories).most_common(5):
    print(f'  {cat}: {cnt}')

## 4. Загрузка модели Gemma 2 2B

In [ ]:
# Для загрузки Gemma нужен HuggingFace токен (примите лицензию на странице модели)
# from huggingface_hub import login
# login()  # раскомментируй, если требуется авторизация

print(f'Загружаем {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'dtype: {dtype}')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map='auto',
)
model.eval()

n_layers    = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
print(f'\nМодель загружена:')
print(f'  Слоёв:       {n_layers}')
print(f'  Hidden size: {hidden_size}')
if DEVICE == 'cuda':
    print(f'  VRAM занято: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 5. Загрузка предобученного SAE (Gemma Scope)

Пробуем три способа загрузки SAE:
1. **sae_lens** — основной (библиотека от EleutherAI/Anthropic community)
2. **Ручная загрузка** с HuggingFace (`google/gemma-scope-2b-pt-res`)
3. **Fallback** — обучаем простой SAE с нуля на собранных hidden states

In [ ]:
class SAEWrapper:
    """Единый интерфейс для SAE вне зависимости от способа загрузки."""
    def __init__(self, encoder_weight, encoder_bias, n_features):
        self.W_enc = encoder_weight.float()  # [hidden_size, n_features]
        self.b_enc = encoder_bias.float()    # [n_features]
        self.n_features = n_features

    @torch.no_grad()
    def encode(self, x):
        """x: [batch, hidden_size] → активации: [batch, n_features]"""
        x = x.float()
        pre = x @ self.W_enc + self.b_enc
        return F.relu(pre)


def load_sae_via_sae_lens(layer_idx, device):
    """Загрузка SAE через sae_lens (Gemma Scope)."""
    from sae_lens import SAE
    sae_id = f'layer_{layer_idx}/width_16k/average_l0_{SAE_L0}'
    sae, cfg_dict, logits_df = SAE.from_pretrained(
        release=SAE_RELEASE,
        sae_id=sae_id,
    )
    sae = sae.to(device)
    # Извлекаем веса в единый формат
    W_enc = sae.W_enc.T.cpu()  # [hidden_size, n_features]
    b_enc = sae.b_enc.cpu()    # [n_features]
    n_features = b_enc.shape[0]
    print(f'  SAE загружен через sae_lens: {n_features} фичей')
    return SAEWrapper(W_enc, b_enc, n_features), sae


def load_sae_manual(layer_idx, device):
    """Ручная загрузка SAE с HuggingFace (без sae_lens)."""
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    repo_id  = 'google/gemma-scope-2b-pt-res'
    filename = f'layer_{layer_idx}/width_16k/average_l0_{SAE_L0}/params.npz'

    try:
        path = hf_hub_download(repo_id=repo_id, filename=filename)
        data = np.load(path)
        W_enc = torch.from_numpy(data['W_enc'])   # [hidden_size, n_features]
        b_enc = torch.from_numpy(data['b_enc'])   # [n_features]
    except Exception:
        # Пробуем safetensors формат
        filename_st = f'layer_{layer_idx}/width_16k/average_l0_{SAE_L0}/params.safetensors'
        path = hf_hub_download(repo_id=repo_id, filename=filename_st)
        data = load_file(path)
        W_enc = data['W_enc']
        b_enc = data['b_enc']

    n_features = b_enc.shape[0]
    print(f'  SAE загружен вручную: {n_features} фичей')
    return SAEWrapper(W_enc.cpu(), b_enc.cpu(), n_features), None


class SimpleSAE(nn.Module):
    """Простой SAE fallback: обучаем с нуля если Gemma Scope недоступен."""
    def __init__(self, input_dim, n_features, l1_coeff=1e-3):
        super().__init__()
        self.n_features = n_features
        self.l1_coeff   = l1_coeff
        self.encoder = nn.Linear(input_dim, n_features)
        self.decoder = nn.Linear(n_features, input_dim, bias=False)
        # Инициализация по Kaiming
        nn.init.kaiming_uniform_(self.encoder.weight)
        nn.init.zeros_(self.encoder.bias)
        # Декодер: начинаем с транспонированного энкодера
        self.decoder.weight.data = self.encoder.weight.data.T.clone()

    def forward(self, x):
        acts = F.relu(self.encoder(x))
        recon = self.decoder(acts)
        recon_loss = F.mse_loss(recon, x)
        l1_loss    = self.l1_coeff * acts.abs().mean()
        return acts, recon_loss + l1_loss

    @torch.no_grad()
    def encode(self, x):
        return F.relu(self.encoder(x.float()))


def train_simple_sae(X_train_np, input_dim, n_features=None, n_epochs=20, device='cpu'):
    """Обучение fallback SAE на hidden states."""
    if n_features is None:
        n_features = input_dim * 4  # overcomplete basis 4x

    print(f'  Обучаем SAE с нуля: {input_dim} → {n_features} фичей')
    sae = SimpleSAE(input_dim, n_features).to(device)
    opt = torch.optim.Adam(sae.parameters(), lr=1e-3)

    X_tensor = torch.from_numpy(X_train_np).float().to(device)
    batch_sz  = 64

    for epoch in range(n_epochs):
        idx   = torch.randperm(len(X_tensor))
        total = 0.0
        for i in range(0, len(X_tensor), batch_sz):
            xb = X_tensor[idx[i:i + batch_sz]]
            opt.zero_grad()
            _, loss = sae(xb)
            loss.backward()
            opt.step()
            # Нормируем столбцы декодера
            with torch.no_grad():
                sae.decoder.weight.data = F.normalize(sae.decoder.weight.data, dim=0)
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f'    Epoch {epoch+1}/{n_epochs}  loss={total:.4f}')

    sae.eval()
    W_enc = sae.encoder.weight.T.detach().cpu()  # [input_dim, n_features]
    b_enc = sae.encoder.bias.detach().cpu()
    wrapper = SAEWrapper(W_enc, b_enc, n_features)
    return wrapper, sae

In [ ]:
# Пробуем загрузить SAE: sae_lens → ручная → fallback
print(f'Загружаем SAE для слоя {BEST_LAYER}...')

sae_wrapper = None
sae_raw     = None
sae_source  = None

# Способ 1: sae_lens
try:
    print('Попытка 1: sae_lens...')
    sae_wrapper, sae_raw = load_sae_via_sae_lens(BEST_LAYER, DEVICE)
    sae_source = 'sae_lens (Gemma Scope)'
except Exception as e:
    print(f'  sae_lens не сработал: {e}')

# Способ 2: ручная загрузка
if sae_wrapper is None:
    try:
        print('Попытка 2: ручная загрузка с HuggingFace...')
        sae_wrapper, sae_raw = load_sae_manual(BEST_LAYER, DEVICE)
        sae_source = 'HuggingFace (ручная загрузка)'
    except Exception as e:
        print(f'  Ручная загрузка не сработала: {e}')

# Способ 3: fallback — тренируем SAE с нуля
# (выполняется после сбора hidden states — см. ячейку ниже)

if sae_wrapper is not None:
    print(f'\nSAE успешно загружен: {sae_source}')
    print(f'  Число фичей: {sae_wrapper.n_features}')
else:
    print('\nSAE не загружен — обучим с нуля после сбора hidden states')

SAE_N_FEATURES = sae_wrapper.n_features if sae_wrapper else None

## 6. Извлечение hidden states

Если есть кэш из probing-части — загружаем. Иначе собираем заново.

In [ ]:
def get_answer_start_idx(input_ids, question_ids):
    ids   = input_ids.tolist()
    q_ids = question_ids.tolist()
    q_len = len(q_ids)
    for i in range(len(ids) - q_len + 1):
        if ids[i:i + q_len] == q_ids:
            return i + q_len
    return len(ids) // 2


@torch.no_grad()
def collect_hidden_states_single_layer(texts, questions, tokenizer, model,
                                        layer_idx, strategy='last_token',
                                        batch_size=BATCH_SIZE):
    """Собирает hidden states только для одного слоя (экономия RAM)."""
    n_total = len(texts)
    result  = np.zeros((n_total, hidden_size), dtype=np.float32)

    for batch_start in tqdm(range(0, n_total, batch_size), desc=f'Hidden states (layer {layer_idx})'):
        batch_texts     = texts[batch_start : batch_start + batch_size]
        batch_questions = questions[batch_start : batch_start + batch_size]

        encoding = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(DEVICE)

        try:
            outputs = model(**encoding, output_hidden_states=True)
        except torch.cuda.OutOfMemoryError:
            print(f'OOM на батче {batch_start}, переключаемся на batch_size=1')
            torch.cuda.empty_cache()
            gc.collect()
            for k, (text, question) in enumerate(zip(batch_texts, batch_questions)):
                enc = tokenizer([text], return_tensors='pt',
                                truncation=True, max_length=MAX_LENGTH).to(DEVICE)
                out = model(**enc, output_hidden_states=True)
                hs  = out.hidden_states[layer_idx][0].float()  # [seq, hidden]
                mask = enc['attention_mask'][0]
                vec  = _aggregate(hs, mask, enc['input_ids'][0], [question], tokenizer, strategy)
                result[batch_start + k] = vec.cpu().numpy()
                del out, enc
                torch.cuda.empty_cache()
            del outputs
            continue

        hs_layer = outputs.hidden_states[layer_idx]  # [batch, seq, hidden]
        for b in range(hs_layer.shape[0]):
            hs   = hs_layer[b].float()
            mask = encoding['attention_mask'][b]
            ids  = encoding['input_ids'][b]
            vec  = _aggregate(hs, mask, ids, batch_questions, tokenizer, strategy, b)
            result[batch_start + b] = vec.cpu().numpy()

        del outputs
        torch.cuda.empty_cache()

    return result


def _aggregate(hs, mask, ids, questions, tokenizer, strategy, b=0):
    """Агрегирует hidden state одного примера по выбранной стратегии."""
    real_len = mask.sum().item()
    hs_real  = hs[:real_len]   # [real_len, hidden]

    if strategy == 'last_token':
        return hs_real[-1]
    elif strategy == 'mean_pooling':
        return hs_real.mean(dim=0)
    elif strategy == 'answer_mean':
        q = questions[b] if b < len(questions) else ''
        prefix   = f'Question: {q}\nAnswer: '
        q_ids    = tokenizer.encode(prefix, add_special_tokens=False)
        ans_start = get_answer_start_idx(ids[:real_len], torch.tensor(q_ids))
        ans_start = min(ans_start, real_len - 1)
        return hs_real[ans_start:].mean(dim=0)
    else:
        return hs_real[-1]

In [ ]:
# Загружаем hidden states: из кэша или собираем заново
if os.path.exists(HIDDEN_STATES_PATH):
    print(f'Загружаем кэш из {HIDDEN_STATES_PATH}...')
    cached = np.load(HIDDEN_STATES_PATH)
    # Кэш хранит все слои: [n_samples, n_layers+1, hidden_size]
    if 'last_token' in cached.files:
        hs_all = cached[AGG_STRATEGY]          # [n_samples, n_layers+1, hidden]
        X_layer = hs_all[:, BEST_LAYER, :]     # [n_samples, hidden]
        print(f'Загружено: слой {BEST_LAYER}, стратегия {AGG_STRATEGY}')
        print(f'Форма: {X_layer.shape}')
    else:
        # Кэш в другом формате — пересобираем для нужного слоя
        print('Формат кэша отличается — собираем hidden states для нужного слоя...')
        X_layer = collect_hidden_states_single_layer(
            texts, questions, tokenizer, model,
            layer_idx=BEST_LAYER, strategy=AGG_STRATEGY
        )
else:
    print('Кэш не найден — собираем hidden states (займёт ~5-10 мин)...')
    X_layer = collect_hidden_states_single_layer(
        texts, questions, tokenizer, model,
        layer_idx=BEST_LAYER, strategy=AGG_STRATEGY
    )

print(f'\nX_layer shape: {X_layer.shape}')  # [1000, 2304]

In [ ]:
# Если SAE не загружен — обучаем с нуля на hidden states
if sae_wrapper is None:
    print('Запускаем fallback: обучаем SAE с нуля...')
    sae_wrapper, sae_raw = train_simple_sae(
        X_layer, input_dim=hidden_size,
        n_features=hidden_size * 4,  # overcomplete 4x
        n_epochs=30, device=DEVICE
    )
    SAE_N_FEATURES = sae_wrapper.n_features
    sae_source = 'Custom SAE (обучен с нуля)'
    print(f'Fallback SAE обучен: {SAE_N_FEATURES} фичей')

print(f'\nИсточник SAE: {sae_source}')
print(f'Число SAE-фичей: {sae_wrapper.n_features}')

## 7. Извлечение SAE-активаций для всех примеров

In [ ]:
if os.path.exists(SAE_ACTIVATIONS_PATH):
    print(f'Загружаем кэш SAE-активаций из {SAE_ACTIVATIONS_PATH}...')
    cached_sae = np.load(SAE_ACTIVATIONS_PATH)
    X_sae  = cached_sae['activations']   # [n_samples, n_features]
    labels_cached = cached_sae['labels']
    print(f'Загружено: {X_sae.shape}')
else:
    print('Извлекаем SAE-активации...')
    n_total    = len(X_layer)
    X_sae      = np.zeros((n_total, sae_wrapper.n_features), dtype=np.float32)

    # Переносим веса SAE на GPU для скорости
    W_enc_gpu = sae_wrapper.W_enc.to(DEVICE)   # [hidden, n_features]
    b_enc_gpu = sae_wrapper.b_enc.to(DEVICE)   # [n_features]

    sae_batch = 64  # больше батч — быстрее (SAE лёгкий)
    for i in tqdm(range(0, n_total, sae_batch), desc='SAE encode'):
        chunk  = torch.from_numpy(X_layer[i:i + sae_batch]).float().to(DEVICE)
        acts   = F.relu(chunk @ W_enc_gpu + b_enc_gpu)
        X_sae[i:i + sae_batch] = acts.cpu().numpy()

    del W_enc_gpu, b_enc_gpu
    torch.cuda.empty_cache()

    np.savez_compressed(
        SAE_ACTIVATIONS_PATH,
        activations=X_sae,
        labels=labels,
    )
    print(f'Сохранено в {SAE_ACTIVATIONS_PATH}')

labels_cached = labels  # используем из текущей сессии

print(f'\nSAE активации: {X_sae.shape}')
sparsity = (X_sae == 0).mean()
print(f'Разреженность: {sparsity:.2%} (среднее L0: {(X_sae > 0).sum(axis=1).mean():.1f})')

## 8. Статистический анализ фичей

Ищем «галлюцинационные» и «truth» фичи через сравнение активаций между классами.

In [ ]:
correct_mask = labels == 1
halluc_mask  = labels == 0

X_correct = X_sae[correct_mask]   # [500, n_features]
X_halluc  = X_sae[halluc_mask]    # [500, n_features]

print('Вычисляем статистику по фичам...')

mean_correct = X_correct.mean(axis=0)   # [n_features]
mean_halluc  = X_halluc.mean(axis=0)    # [n_features]
diff_means   = mean_halluc - mean_correct  # > 0 → фича активна на галлюцинациях

# Частота активации (>0) для каждого класса
freq_correct = (X_correct > 0).mean(axis=0)
freq_halluc  = (X_halluc  > 0).mean(axis=0)

print('Mann-Whitney U тест для каждой фичи...')
from scipy.stats import mannwhitneyu

n_features = X_sae.shape[1]
p_values   = np.zeros(n_features)
u_stats    = np.zeros(n_features)

# Батчами по 1000 фичей для скорости
chunk_size = 1000
for start in tqdm(range(0, n_features, chunk_size), desc='Mann-Whitney'):
    end = min(start + chunk_size, n_features)
    for feat_idx in range(start, end):
        a = X_correct[:, feat_idx]
        b = X_halluc[:, feat_idx]
        if a.sum() == 0 and b.sum() == 0:
            p_values[feat_idx]  = 1.0
            u_stats[feat_idx]   = 0.0
        else:
            u, p = mannwhitneyu(a, b, alternative='two-sided')
            p_values[feat_idx] = p
            u_stats[feat_idx]  = u

# Поправка Бонферрони (или FDR)
from statsmodels.stats.multitest import multipletests
try:
    _, p_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
except ImportError:
    p_corrected = np.minimum(p_values * n_features, 1.0)  # Bonferroni

significant_mask = p_corrected < 0.05
print(f'\nСтатистически значимых фичей (FDR < 0.05): {significant_mask.sum()}')

In [ ]:
# Формируем DataFrame со всеми фичами
feature_df = pd.DataFrame({
    'feature_idx':     np.arange(n_features),
    'mean_correct':    mean_correct,
    'mean_halluc':     mean_halluc,
    'diff_means':      diff_means,           # + = halluc выше, - = correct выше
    'abs_diff':        np.abs(diff_means),
    'freq_correct':    freq_correct,
    'freq_halluc':     freq_halluc,
    'freq_diff':       freq_halluc - freq_correct,
    'p_value':         p_values,
    'p_corrected':     p_corrected,
    'significant':     significant_mask,
})

# Top-20 «галлюцинационных» фичей (активнее на галлюцинациях)
top_halluc = feature_df[feature_df['diff_means'] > 0].nlargest(20, 'abs_diff')
# Top-20 «truth» фичей (активнее на корректных ответах)
top_truth  = feature_df[feature_df['diff_means'] < 0].nlargest(20, 'abs_diff')

print('Top-10 HALLUCINATION фичей:')
print(top_halluc[['feature_idx', 'mean_halluc', 'mean_correct', 'diff_means',
                   'freq_halluc', 'freq_correct', 'p_corrected']].head(10).to_string(index=False))
print()
print('Top-10 TRUTH фичей:')
print(top_truth[['feature_idx', 'mean_halluc', 'mean_correct', 'diff_means',
                  'freq_halluc', 'freq_correct', 'p_corrected']].head(10).to_string(index=False))

## 9. SAE-based классификатор

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sae, labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED,
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Probe на ВСЕХ SAE-фичах
print('LogisticRegression на всех SAE-фичах...')
clf_full = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED, C=1.0)
clf_full.fit(X_train_sc, y_train)
acc_full = accuracy_score(y_test, clf_full.predict(X_test_sc))
print(f'  Accuracy (all {n_features} features): {acc_full:.4f}')

# Probe на Top-K фичах (по abs_diff)
top_k_values = [10, 20, 50, 100, 200, 500]
top_feature_indices = feature_df.nlargest(max(top_k_values), 'abs_diff')['feature_idx'].values

acc_topk = {}
for k in top_k_values:
    feat_k = top_feature_indices[:k]

    X_tr_k = X_train[:, feat_k]
    X_te_k = X_test[:, feat_k]

    sc_k = StandardScaler()
    X_tr_k = sc_k.fit_transform(X_tr_k)
    X_te_k = sc_k.transform(X_te_k)

    clf_k = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED, C=1.0)
    clf_k.fit(X_tr_k, y_train)
    acc_k = accuracy_score(y_test, clf_k.predict(X_te_k))
    acc_topk[k] = acc_k
    print(f'  Top-{k:4d} features: {acc_k:.4f}')

print(f'\nBaseline (Self-Consistency EN): {BASELINE_ACC:.4f}')
print(f'SAE probe (all):                {acc_full:.4f}')
best_topk_k   = max(acc_topk, key=acc_topk.get)
best_topk_acc = acc_topk[best_topk_k]
print(f'SAE probe (top-{best_topk_k}):            {best_topk_acc:.4f}')

## 10. Визуализации

In [ ]:
# === График 1: Bar chart — Top-20 фичей по разнице активаций ===

n_top = 20
top_h = feature_df.nlargest(n_top, 'diff_means').copy()
top_t = feature_df.nsmallest(n_top, 'diff_means').copy()
combined = pd.concat([top_h, top_t], ignore_index=True)
combined = combined.sort_values('diff_means', ascending=True)

colors_bar = ['#E53935' if d > 0 else '#43A047' for d in combined['diff_means']]

fig, ax = plt.subplots(figsize=(12, 9))
bars = ax.barh(
    [f'Feature {int(i)}' for i in combined['feature_idx']],
    combined['diff_means'],
    color=colors_bar,
    edgecolor='white',
    linewidth=0.4,
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Разница средних активаций (hallucination − correct)', fontsize=12)
ax.set_title(f'Top-{n_top} галлюцинационных и {n_top} truth-фичей\n'
             f'(Gemma 2 2B, layer {BEST_LAYER}, {sae_source})',
             fontsize=13, fontweight='bold')

halluc_patch = mpatches.Patch(color='#E53935', label='Hallucination фичи (активнее на галлюцинациях)')
truth_patch  = mpatches.Patch(color='#43A047', label='Truth фичи (активнее на корректных ответах)')
ax.legend(handles=[halluc_patch, truth_patch], fontsize=10, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('sae_top_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: sae_top_features.png')

In [ ]:
# === График 2: Distribution plots — распределение активаций топ-6 фичей ===

# 3 галлюцинационные + 3 truth
plot_features = list(top_halluc['feature_idx'].values[:3]) + list(top_truth['feature_idx'].values[:3])
plot_labels   = ['Halluc'] * 3 + ['Truth'] * 3
feat_types    = ['Hallucination'] * 3 + ['Truth'] * 3

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Распределение SAE-активаций: informative фичи\n'
             '(синий = correct, красный = hallucination)',
             fontsize=13, fontweight='bold')

for ax, feat_idx, feat_type in zip(axes.flat, plot_features, feat_types):
    vals_corr  = X_sae[correct_mask, feat_idx]
    vals_halluc = X_sae[halluc_mask,  feat_idx]

    # KDE только если есть ненулевые значения
    if vals_corr.max() > 0:
        sns.kdeplot(vals_corr,  ax=ax, color='#1E88E5', label='Correct',       fill=True, alpha=0.35)
    else:
        ax.axvline(0, color='#1E88E5', linewidth=2, label='Correct (≈0)')

    if vals_halluc.max() > 0:
        sns.kdeplot(vals_halluc, ax=ax, color='#E53935', label='Hallucination', fill=True, alpha=0.35)
    else:
        ax.axvline(0, color='#E53935', linewidth=2, label='Hallucination (≈0)')

    d_str = f"{feature_df.loc[feature_df['feature_idx']==feat_idx, 'diff_means'].values[0]:+.3f}"
    ax.set_title(f'Feature {feat_idx}\n({feat_type}, Δ={d_str})', fontsize=10)
    ax.set_xlabel('Активация', fontsize=9)
    ax.set_ylabel('Плотность', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('sae_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: sae_distributions.png')

In [ ]:
# === График 3: Accuracy vs число фичей ===

k_values  = list(acc_topk.keys()) + [n_features]
acc_values = list(acc_topk.values()) + [acc_full]
k_labels   = [str(k) for k in top_k_values] + ['All\n' + str(n_features)]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(range(len(k_values)), acc_values, marker='o', color='#1565C0',
        linewidth=2, markersize=8, zorder=3)

for x, y, lbl in zip(range(len(k_values)), acc_values, k_labels):
    ax.annotate(f'{y:.3f}', xy=(x, y), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=9, color='#1565C0')

ax.axhline(BASELINE_ACC, color='gray', linestyle='--', linewidth=1.5,
           label=f'Baseline Self-Cons. EN = {BASELINE_ACC:.2%}')
ax.axhline(PART2_BEST_LINEAR, color='#43A047', linestyle=':', linewidth=1.5,
           label=f'Linear probe на hidden states = {PART2_BEST_LINEAR:.2%}')

ax.set_xticks(range(len(k_values)))
ax.set_xticklabels(k_labels, fontsize=10)
ax.set_xlabel('Число top-K SAE-фичей', fontsize=12)
ax.set_ylabel('Accuracy (test)', fontsize=12)
ax.set_title('Точность SAE-probe vs число используемых фичей', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig('sae_acc_vs_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: sae_acc_vs_features.png')

In [ ]:
# === График 4: Heatmap корреляций между top-20 фичами ===

top20_idx = feature_df.nlargest(20, 'abs_diff')['feature_idx'].values
X_top20   = X_sae[:, top20_idx]

corr_matrix = np.corrcoef(X_top20.T)  # [20, 20]
feat_labels = [f'F{int(i)}' for i in top20_idx]

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    xticklabels=feat_labels,
    yticklabels=feat_labels,
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7},
    linewidths=0.3,
    ax=ax,
)
ax.set_title('Матрица корреляций между top-20 SAE-фичами\n'
             '(чем ближе к 0 — тем независимее фичи)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('sae_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: sae_correlation_heatmap.png')

In [ ]:
# === График 5: Итоговое сравнение всех методов (части 1 + 2 + 3) ===

all_methods = dict(PART1_RESULTS)
all_methods['Probing\n(linear probe)']      = PART2_BEST_LINEAR
all_methods[f'SAE probe\n(all {n_features} feat)'] = acc_full
all_methods[f'SAE probe\n(top-{best_topk_k} feat)']  = best_topk_acc

methods_sorted = sorted(all_methods.items(), key=lambda x: x[1])
names, accs = zip(*methods_sorted)

# Цвета по группам методов
def method_color(name):
    if 'SAE' in name:     return '#E53935'
    if 'Probing' in name: return '#1E88E5'
    if 'Reference' in name: return '#8E24AA'
    return '#78909C'

bar_colors = [method_color(n) for n in names]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(names, accs, color=bar_colors, edgecolor='white', linewidth=0.5)

ax.axhline(BASELINE_ACC, color='black', linestyle=':', linewidth=1.5,
           label=f'Baseline (Self-Cons. EN = {BASELINE_ACC:.2%})', zorder=5)

for bar, val in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.1%}',
        ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

legend_patches = [
    mpatches.Patch(color='#78909C', label='Часть 1: LLM-as-a-Judge'),
    mpatches.Patch(color='#8E24AA', label='Часть 1: Reference-based'),
    mpatches.Patch(color='#1E88E5', label='Часть 2: Probing (hidden states)'),
    mpatches.Patch(color='#E53935', label='Часть 3: SAE probe'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='upper left')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.set_title('Сравнение всех методов Hiromi\n(Части 1 + 2 + 3)',
             fontsize=14, fontweight='bold')
ax.tick_params(axis='x', labelsize=8)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sae_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: sae_final_comparison.png')

## 11. Интерпретация топ-фичей + ссылки на Neuronpedia

In [ ]:
# Топ-10 самых информативных фичей (по abs_diff)
top10_feats = feature_df.nlargest(10, 'abs_diff')

print('=' * 70)
print('ИНТЕРПРЕТАЦИЯ TOP-10 SAE-ФИЧЕЙ')
print('=' * 70)

for rank, (_, row) in enumerate(top10_feats.iterrows(), 1):
    feat_idx  = int(row['feature_idx'])
    feat_type = 'HALLUCINATION' if row['diff_means'] > 0 else 'TRUTH'

    # Примеры текстов с максимальной активацией этой фичи
    activations_feat = X_sae[:, feat_idx]
    top5_idxs = np.argsort(activations_feat)[::-1][:5]

    # Ссылка на Neuronpedia
    neuronpedia_url = f'https://www.neuronpedia.org/gemma-2-2b/{BEST_LAYER}/{feat_idx}'

    print(f'\n[{rank}] Feature {feat_idx} ({feat_type})')
    print(f'    Δ mean = {row["diff_means"]:+.4f}  '
          f'  freq_halluc={row["freq_halluc"]:.3f}  '
          f'freq_correct={row["freq_correct"]:.3f}  '
          f'p_adj={row["p_corrected"]:.2e}')
    print(f'    Neuronpedia: {neuronpedia_url}')
    print(f'    Топ-5 текстов с макс. активацией:')

    for top_rank, idx in enumerate(top5_idxs, 1):
        act_val   = activations_feat[idx]
        label_str = 'correct' if labels[idx] == 1 else 'hallucination'
        text_short = texts[idx][:120].replace('\n', ' ')
        print(f'      {top_rank}. [{label_str}] act={act_val:.3f}: {text_short}...')

In [ ]:
# Сохраняем таблицу с ссылками в HTML для удобного просмотра
top10_display = top10_feats[['feature_idx', 'diff_means', 'freq_halluc',
                               'freq_correct', 'p_corrected']].copy()
top10_display['type'] = top10_display['diff_means'].apply(
    lambda x: 'HALLUCINATION' if x > 0 else 'TRUTH'
)
top10_display['neuronpedia'] = top10_display['feature_idx'].apply(
    lambda i: f'https://www.neuronpedia.org/gemma-2-2b/{BEST_LAYER}/{int(i)}'
)
top10_display = top10_display.round(4)
print(top10_display.to_string(index=False))

## 12. Анализ по категориям TruthfulQA

In [ ]:
# Лучший SAE-классификатор (top-K)
feat_topk = top_feature_indices[:best_topk_k]
X_topk_all = X_sae[:, feat_topk]
sc_topk    = StandardScaler()
X_topk_all_sc = sc_topk.fit_transform(X_topk_all)

# Переобучаем на всём train-сете
train_idx = list(range(len(labels)))
X_tr_topk = sc_topk.fit_transform(X_sae[train_idx, :][:, feat_topk])
clf_topk_full = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED, C=1.0)
clf_topk_full.fit(X_tr_topk, labels[train_idx])

# Предсказания на всей выборке
preds_topk = clf_topk_full.predict(X_topk_all_sc)

# Accuracy по категориям
cat_results = []
all_cats = sorted(set(categories))

for cat in all_cats:
    cat_idx  = [i for i, c in enumerate(categories) if c == cat]
    if len(cat_idx) < 5:
        continue  # пропускаем категории с малым числом примеров
    y_cat    = labels[cat_idx]
    pred_cat = preds_topk[cat_idx]
    acc_cat  = accuracy_score(y_cat, pred_cat)
    n_halluc = (y_cat == 0).sum()
    n_corr   = (y_cat == 1).sum()
    cat_results.append({
        'category': cat,
        'n_total':  len(cat_idx),
        'n_correct': n_corr,
        'n_halluc': n_halluc,
        'acc_sae':  acc_cat,
    })

df_cats = pd.DataFrame(cat_results).sort_values('acc_sae', ascending=False)
print('Accuracy SAE-probe по категориям TruthfulQA:')
print(df_cats.to_string(index=False))

In [ ]:
# Визуализация accuracy по категориям
fig, ax = plt.subplots(figsize=(14, 6))

cats_sorted = df_cats.sort_values('acc_sae', ascending=True)

bar_colors_cat = [
    '#E53935' if acc < BASELINE_ACC else '#43A047'
    for acc in cats_sorted['acc_sae']
]

ax.barh(cats_sorted['category'], cats_sorted['acc_sae'],
        color=bar_colors_cat, edgecolor='white', linewidth=0.3)

ax.axvline(BASELINE_ACC, color='black', linestyle=':', linewidth=1.5,
           label=f'Baseline = {BASELINE_ACC:.2%}')
ax.axvline(acc_topk.get(best_topk_k, acc_full), color='#1565C0',
           linestyle='--', linewidth=1.5,
           label=f'SAE probe overall = {acc_topk.get(best_topk_k, acc_full):.2%}')

for i, (_, row) in enumerate(cats_sorted.iterrows()):
    ax.text(row['acc_sae'] + 0.005, i, f'{row["acc_sae"]:.1%}\n(n={row["n_total"]})',
            va='center', fontsize=8)

ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title(f'Accuracy SAE-probe по категориям TruthfulQA\n'
             f'(top-{best_topk_k} фичей)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0.3, 1.05)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('sae_category_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: sae_category_accuracy.png')

## 13. Итоговое сравнение всех методов

In [ ]:
# Итоговая сравнительная таблица
summary_rows = []

for method, acc in PART1_RESULTS.items():
    summary_rows.append({'Метод': method, 'Accuracy': acc, 'Часть': 'Часть 1'})

summary_rows.append({'Метод': 'Linear Probe (лучший слой)', 'Accuracy': PART2_BEST_LINEAR, 'Часть': 'Часть 2'})

for k, acc in acc_topk.items():
    summary_rows.append({'Метод': f'SAE Probe (top-{k})', 'Accuracy': acc, 'Часть': 'Часть 3'})
summary_rows.append({'Метод': f'SAE Probe (all {n_features})', 'Accuracy': acc_full, 'Часть': 'Часть 3'})

df_summary = pd.DataFrame(summary_rows).sort_values('Accuracy', ascending=False)
df_summary['Accuracy %'] = df_summary['Accuracy'].map('{:.2%}'.format)
df_summary['vs Baseline'] = (df_summary['Accuracy'] - BASELINE_ACC).map('{:+.2%}'.format)

print('=' * 65)
print('ИТОГОВОЕ СРАВНЕНИЕ ВСЕХ МЕТОДОВ HIROMI')
print('=' * 65)
print(df_summary[['Метод', 'Часть', 'Accuracy %', 'vs Baseline']].to_string(index=False))

## 14. Выводы

In [ ]:
print('=' * 65)
print('ВЫВОДЫ — SAE-анализ галлюцинаций (Hiromi, часть 3)')
print('=' * 65)

sae_beat_baseline = acc_full > BASELINE_ACC
topk_beat_baseline = best_topk_acc > BASELINE_ACC

print(f"""
1. SAE-АКТИВАЦИИ И ГАЛЛЮЦИНАЦИИ
   Разреженность SAE: {sparsity:.1%} нулевых активаций
   Среднее L0:        {(X_sae > 0).sum(axis=1).mean():.1f} активных фичей на текст
   Значимых фичей:    {significant_mask.sum()} из {n_features} (FDR < 0.05)

2. КАЧЕСТВО КЛАССИФИКАЦИИ
   SAE probe (all):      {acc_full:.4f} ({'+' if sae_beat_baseline else ''}{acc_full - BASELINE_ACC:+.4f} vs baseline)
   SAE probe (top-{best_topk_k}):  {best_topk_acc:.4f} ({'+' if topk_beat_baseline else ''}{best_topk_acc - BASELINE_ACC:+.4f} vs baseline)
   Linear probe (ч.2):  {PART2_BEST_LINEAR:.4f} ({PART2_BEST_LINEAR - BASELINE_ACC:+.4f} vs baseline)

3. ИНТЕРПРЕТИРУЕМОСТЬ
   Всего {len(top10_feats)} топ-фичей можно исследовать на Neuronpedia:
   {chr(10).join(f'   Feature {int(row["feature_idx"])}: https://www.neuronpedia.org/gemma-2-2b/{BEST_LAYER}/{int(row["feature_idx"])}'
                  for _, row in top10_feats.head(3).iterrows())}

4. КЛЮЧЕВОЙ ВЫВОД
   SAE позволяет не просто детектировать галлюцинации, но и
   найти конкретные интерпретируемые нейронные фичи, ответственные
   за это разделение. Это важнее accuracy — мы видим «механизм»
   галлюцинаций изнутри модели.
""")